In [ ]:
import numpy as np
import pandas as pd
import os
from time import time
from pulp import *

In [ ]:
def save_experiment_json(exp_id, prob, x, P, alpha, beta, n_limit, penalties, matrix_file):
    solve_time = getattr(prob, 'solutionTime', 0)
    solver_name = getattr(prob, 'usedSolver', "CBC")
    status = LpStatus[prob.status]
    
    selected_duties = [j for j, var in x.items() if value(var) > 0.5]
    cancelled_pows = [i for i, var in P.items() if value(var) > 0.5]
    total_penalty = sum(penalties[i] for i in cancelled_pows)

    experiment_data = {
        "metadata": {
            "exp_id": exp_id,
            "matrix_used": matrix_file
        },
        "input_config": {
            "alpha": alpha,
            "beta": beta,
            "N_limit": n_limit,
            "penalty_vector": penalties
        },
        "output_results": {
            "status": status,
            "solver": solver_name,
            "runtime_sec": round(solve_time, 4),
            "total_cost": value(prob.objective) if status == 'Optimal' else None,
            "drivers_used": len(selected_duties),
            "total_penalty": total_penalty,
            "cancelled_count": len(cancelled_pows),
            "selected_duty_indices": selected_duties,
            "cancelled_pow_indices": cancelled_pows
        }
    }

    json_output = json.dumps(experiment_data, indent=4)
    
    for key in ["penalty_vector", "selected_duty_indices", "cancelled_pow_indices"]:
        json_output = re.sub(
            rf'("{key}":\s*)\[\s*(.*?)\s*\]',
            lambda m: m.group(1) + '[' + re.sub(r'\s+', ' ', m.group(2)) + ']',
            json_output,
            flags=re.DOTALL
        )

    if not os.path.exists("results"):
        os.makedirs("results")

    filename = f"results/{exp_id}_full_report.json"
    with open(filename, "w", encoding="utf-8") as f:
        f.write(json_output)
        
    return experiment_data

In [ ]:
# undone
def load_matrix(file_name):
    df = pd.read_csv(file_name, index_col=0)
    
    matrix_A = df.values
    
    m, n = matrix_A.shape
    
    return matrix_A, m, n

In [ ]:
def build_and_solve_model(matrix_A, alpha, beta, n_limit, penalties):
    """
    Establish and solve the Set Covering model.
    """
    m, n = matrix_A.shape
    
    # --- 1. Initialize the model ---
    prob = LpProblem("Driver_Scheduling", LpMinimize)
    
    # --- 2. Define model variables ---
    x = LpVariable.dicts("Duty", range(n), cat='Binary')
    P = LpVariable.dicts("Penalty", range(m), cat='Binary')
    
    # --- 3. Objective function ---
    # min (alpha * sum x_j + beta * sum Pi * Ci)
    prob += (alpha * lpSum([x[j] for j in range(n)]) + 
             beta * lpSum([P[i] * penalties[i] for i in range(m)]))
    
    # --- 4. Constraints ---
    # A. Cover or Abandon: sum(x_ij) + Pi >= 1
    # Preprocessing: Find out which Duty j covers each PoW i to speed up modeling
    for i in range(m):
        covered_by_duties = [x[j] for j in range(n) if matrix_A[i][j] == 1]
        prob += lpSum(covered_by_duties) + P[i] >= 1
        
    # B. Max avaible drivers: sum(x_j) <= N
    prob += lpSum([x[j] for j in range(n)]) <= n_limit
    
    # --- 5. Solve ---
    # 如果有 Gurobi，可以換成 prob.solve(GUROBI_CMD())
    start_solve = time()
    prob.solve(PULP_CBC_CMD(msg=0))
    
    # The solution time is recorded in the prob
    prob.solutionTime = time() - start_solve
    
    return prob, x, P

In [ ]:
def run_experiment_pipeline(exp_id, alpha, beta, n_limit, matrix_file, penalties):

    # --- 1. Read data ---
    matrix_A, m, n = load_matrix(matrix_file)
    
    # --- 2. Establish and solve the model ---
    prob, x, P = build_and_solve_model(matrix_A, alpha, beta, n_limit, penalties)

    # --- 3. Call the saving function ---
    report = save_experiment_json(
        exp_id, prob, x, P, 
        alpha, beta, n_limit, penalties, matrix_file
    )
    
    # --- 4. Show brief results ---
    status = report['output_results']['status']
    total_cost = report['output_results']['total_cost']
    print(f"Experiment {exp_id} done！Status: {status}, Total cost: {total_cost}")

In [ ]:
def run_stress_test_pipeline(alpha, beta, matrix_file, penalties, test_plans):

    # --- 1. Read data ---
    matrix_A, m, n = load_matrix(matrix_file)
    
    # --- 2. Find N_base ---
    print(f"--- 正在計算基準人力需求 (100% Coverage) ---")
    base_prob, base_x, base_P = build_and_solve_model(matrix_A, alpha, beta, n_limit=n, penalties=penalties)
    
    if LpStatus[base_prob.status] != 'Optimal':
        print("基準實驗未找到最優解，請檢查模型。")
        return

    # Calculate the actual number of drivers used
    n_base = sum(value(base_x[j]) for j in range(n) if value(base_x[j]) > 0.5)
    print(f"基準人力需求 N_base = {n_base}\n")

    # --- 3. Define all N values ​​to be tested ---    
    for pct in test_plans:
        current_n_limit = int(n_base * pct)
        
        # Dynamically generate experiment IDs
        clean_name = os.path.basename(matrix_file).replace('.csv', '')

        exp_id = f"STRESS_A{alpha}_B{beta}_N{current_n_limit}_{pct*100:.0f}pct_{clean_name}"
        
        print(f">>> 執行實驗: {exp_id} (人力: {current_n_limit}, 比例: {pct*100:.0f}%)")
        
        # 呼叫你原本寫好的單次實驗 Pipeline
        run_experiment_pipeline(
            exp_id=exp_id,
            alpha=alpha,
            beta=beta,
            n_limit=current_n_limit,
            matrix_file=matrix_file,
            penalties=penalties
        )
    
    print("\n✅ 所有壓力測試實驗已完成！")

In [ ]:
EXP_ID = "EXP_test" 
alpha = 1.0     # Driver weight
beta = 1.0      # Penalty weight
N = 10          # Max avaible drivers
target_percentages = [0.85, 0.75, 0.60]
# penalties = []  # Penalty coefficient list
penalties = [100 if i < 30 else 1 for i in range(100)]
matrix_file = 'mock_duty_matrix_100_500.csv' # Input matrix

In [ ]:
run_experiment_pipeline(EXP_ID, alpha, beta, N, matrix_file, penalties)

In [ ]:
run_stress_test_pipeline(alpha, beta, matrix_file, penalties, target_percentages)